# 🔢 NY Lottery Frequency Analysis

This notebook performs detailed frequency analysis on lottery numbers.

## Topics Covered
- Number frequency distributions
- Hot and cold number analysis
- Pair and combination analysis
- Sum and odd/even distributions

## ⚠️ Disclaimer
Frequency analysis shows historical patterns only. Every number has equal probability in each draw. Past results do not influence future outcomes.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
import sys

# Add project root to path
sys.path.insert(0, '..')

from src.analytics.frequency import FrequencyAnalyzer
from src.analytics.hotcold import HotColdAnalyzer

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Initialize analyzers
freq_analyzer = FrequencyAnalyzer('data/lottery_star.db')
hotcold_analyzer = HotColdAnalyzer('data/lottery_star.db')

print("Analyzers initialized!")

## 1. Mega Millions Number Frequency

In [ ]:
# Get frequency for Mega Millions
mm_freq = freq_analyzer.get_set_draw_frequencies('mega_millions')
print(f"Total unique numbers drawn: {len(mm_freq)}")
print("\nTop 10 most frequent numbers:")
print(mm_freq.head(10))

In [ ]:
# Visualize Mega Millions frequency
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of top 20
ax1 = axes[0]
top20 = mm_freq.head(20)
ax1.barh(top20['number'].astype(str), top20['count'], color='steelblue')
ax1.set_xlabel('Times Drawn')
ax1.set_title('Top 20 Most Frequent Mega Millions Numbers')
ax1.invert_yaxis()

# Full distribution
ax2 = axes[1]
ax2.bar(mm_freq['number'], mm_freq['count'], alpha=0.7, color='coral')
ax2.axhline(mm_freq['count'].mean(), color='red', linestyle='--', label=f"Mean: {mm_freq['count'].mean():.1f}")
ax2.set_xlabel('Number')
ax2.set_ylabel('Frequency')
ax2.set_title('Full Number Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Mega Ball frequency
mm_bonus = freq_analyzer.get_bonus_frequencies('mega_millions')
print("Mega Ball Frequency:")
print(mm_bonus.head(10))

plt.figure(figsize=(12, 4))
plt.bar(mm_bonus['bonus'], mm_bonus['count'], color='gold')
plt.xlabel('Mega Ball')
plt.ylabel('Frequency')
plt.title('Mega Ball Frequency Distribution')
plt.show()

## 2. Powerball Number Frequency

In [ ]:
# Powerball analysis
pb_freq = freq_analyzer.get_set_draw_frequencies('powerball')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Main numbers
axes[0].bar(pb_freq['number'], pb_freq['count'], alpha=0.7, color='royalblue')
axes[0].axhline(pb_freq['count'].mean(), color='red', linestyle='--')
axes[0].set_xlabel('Number')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Powerball Main Numbers')

# Powerball
pb_bonus = freq_analyzer.get_bonus_frequencies('powerball')
axes[1].bar(pb_bonus['bonus'], pb_bonus['count'], color='crimson')
axes[1].set_xlabel('Powerball')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Powerball (Red Ball) Frequency')

plt.tight_layout()
plt.show()

## 3. Hot and Cold Numbers Analysis

In [ ]:
# Hot/Cold for different time windows
windows = [30, 90, 180, 365]
game = 'powerball'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, days in enumerate(windows):
    hc_data = hotcold_analyzer.get_hot_cold_numbers(game, days=days, top_n=8)
    
    hot = hc_data['hot']
    cold = hc_data['cold']
    
    ax = axes[i]
    
    x = np.arange(8)
    width = 0.35
    
    if not hot.empty and not cold.empty:
        ax.bar(x - width/2, hot['count'].head(8), width, label='Hot', color='red', alpha=0.7)
        ax.bar(x + width/2, cold['count'].head(8), width, label='Cold', color='blue', alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels([f"H:{h}\nC:{c}" for h, c in zip(hot['number'].head(8), cold['number'].head(8))], fontsize=8)
    
    ax.set_title(f'Last {days} Days')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle(f'{game.replace("_", " ").title()} - Hot vs Cold Numbers by Time Window', fontsize=14)
plt.tight_layout()
plt.show()

print("\n⚠️ Remember: 'Hot' and 'cold' status has NO predictive value!")

## 4. Number Pair Analysis

In [ ]:
# Most common pairs
for game in ['mega_millions', 'powerball', 'take5']:
    pairs = freq_analyzer.get_pair_frequencies(game, top_n=10)
    print(f"\n{game.replace('_', ' ').title()} - Top 10 Number Pairs:")
    pairs['pair'] = pairs.apply(lambda r: f"{int(r['num1'])}-{int(r['num2'])}", axis=1)
    print(pairs[['pair', 'count']].to_string(index=False))

In [ ]:
# Visualize Mega Millions pairs
mm_pairs = freq_analyzer.get_pair_frequencies('mega_millions', top_n=15)
mm_pairs['pair'] = mm_pairs.apply(lambda r: f"{int(r['num1'])}-{int(r['num2'])}", axis=1)

plt.figure(figsize=(12, 6))
plt.barh(mm_pairs['pair'], mm_pairs['count'], color='teal')
plt.xlabel('Times Appearing Together')
plt.ylabel('Number Pair')
plt.title('Mega Millions - Most Common Number Pairs')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Sum Distribution Analysis

In [ ]:
# Sum distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

games = ['mega_millions', 'powerball', 'take5', 'ny_lotto']
colors = ['steelblue', 'crimson', 'green', 'purple']

for ax, game, color in zip(axes.flatten(), games, colors):
    sum_dist = freq_analyzer.get_sum_distribution(game)
    if not sum_dist.empty:
        ax.hist(sum_dist['sum'].repeat(sum_dist['count']), bins=30, color=color, alpha=0.7, edgecolor='black')
        mean_sum = (sum_dist['sum'] * sum_dist['count']).sum() / sum_dist['count'].sum()
        ax.axvline(mean_sum, color='red', linestyle='--', label=f'Mean: {mean_sum:.1f}')
        ax.set_xlabel('Sum of Numbers')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{game.replace("_", " ").title()}')
        ax.legend()

plt.suptitle('Distribution of Number Sums by Game', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Odd/Even Distribution

In [ ]:
# Odd/Even analysis
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, game in zip(axes.flatten(), ['mega_millions', 'powerball', 'take5', 'cash4life']):
    oe_dist = freq_analyzer.get_odd_even_distribution(game)
    if not oe_dist.empty:
        oe_dist['label'] = oe_dist.apply(lambda r: f"{int(r['odd_count'])}O-{int(r['even_count'])}E", axis=1)
        ax.pie(oe_dist['frequency'], labels=oe_dist['label'], autopct='%1.1f%%', 
               colors=sns.color_palette('Set2', len(oe_dist)))
        ax.set_title(f'{game.replace("_", " ").title()}')

plt.suptitle('Odd/Even Number Distribution by Game', fontsize=14)
plt.tight_layout()
plt.show()

print("\nNote: Most draws have a balanced odd/even split, as expected from random selection.")

## Summary

This frequency analysis revealed:

1. **Number Frequencies**: While some numbers appear more often historically, these differences are within expected statistical variance for random draws.

2. **Hot/Cold Patterns**: Numbers cycle between hot and cold status based on recent draws. This has NO predictive value.

3. **Pair Analysis**: Common pairs reflect random co-occurrence, not meaningful relationships.

4. **Sum Distribution**: Sums follow expected bell-curve distributions centered around the mathematical mean.

5. **Odd/Even Split**: Balanced distributions as expected from true randomness.

**Key Takeaway**: All patterns observed are consistent with random, independent draws. They cannot be used to predict future outcomes.